# 8 — E4: Well-Formedness (Optional)

**Hypothesis H5:** Models distinguish well-formed from ill-formed code via dedicated validity neurons.

Compares universal circuits from well-formed prompts (3_object_masks) against circuits
from malformed prompts (3_malformed_masks) to find validity and error-detection neurons.

**Prerequisites:** `1C_malformed_prompts.ipynb` must be run first, followed by extraction (2)
and threshold sweep (3) on the malformed prompts.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas", "matplotlib", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns

COMBOS = [
    {"lang": "P", "model": "QW", "label": "Python/Qwen"},
    # {"lang": "P", "model": "DS", "label": "Python/DeepSeek"},
    {"lang": "R", "model": "QW", "label": "Rust/Qwen"},
    # {"lang": "R", "model": "DS", "label": "Rust/DeepSeek"},
]

EPSILON = 0.001
CONSISTENCY = 0.8
N_LAYERS = 28

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"



print(f"Settings: eps={EPSILON}, cons={CONSISTENCY}")

In [ ]:
# Cell 2 – Load well-formed and malformed masks

wellformed = {}   # (lang, model) -> {concept: {layer: mask}}
malformed = {}    # (lang, model) -> {concept: {layer: mask}}

for combo in COMBOS:
    prefix = f"{combo['lang']}_{combo['model']}_"
    key = (combo["lang"], combo["model"])

    # Well-formed (object masks)
    wf_path = os.path.join(DATA_DIR, f"{prefix}3_object_masks_eps{EPSILON}_cons{CONSISTENCY}.h5")
    if not os.path.exists(wf_path):
        print(f"  WARNING: missing well-formed masks for {combo['label']}")
        continue

    wf = {}
    with h5py.File(wf_path, "r") as f:
        if "universal_masks" in f:
            um = f["universal_masks"]
            for lid in range(N_LAYERS):
                lk = f"layer_{lid}"
                if lk not in um:
                    continue
                for ds_name in um[lk]:
                    if ds_name not in wf:
                        wf[ds_name] = {}
                    wf[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
    wellformed[key] = wf
    print(f"  {combo['label']} well-formed: {len(wf)} concepts")

    # Malformed
    mf_path = os.path.join(DATA_DIR, f"{prefix}3_malformed_masks_eps{EPSILON}_cons{CONSISTENCY}.h5")
    if not os.path.exists(mf_path):
        print(f"  WARNING: missing malformed masks for {combo['label']} — run 1C first")
        continue

    mf = {}
    with h5py.File(mf_path, "r") as f:
        if "universal_masks" in f:
            um = f["universal_masks"]
            for lid in range(N_LAYERS):
                lk = f"layer_{lid}"
                if lk not in um:
                    continue
                for ds_name in um[lk]:
                    if ds_name not in mf:
                        mf[ds_name] = {}
                    mf[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
    malformed[key] = mf
    print(f"  {combo['label']} malformed: {len(mf)} concepts")

In [ ]:
# Cell 3 – Compute validity, error-detection, and shared neurons

results = []

for key in wellformed:
    if key not in malformed:
        continue
    lang, model = key
    wf = wellformed[key]
    mf = malformed[key]

    testable = sorted(set(wf.keys()) & set(mf.keys()))

    for concept in testable:
        for lid in range(N_LAYERS):
            A = wf[concept].get(lid)
            M = mf[concept].get(lid)
            if A is None or M is None:
                continue

            validity = np.where(A & ~M)[0]       # well-formed only
            error_det = np.where(~A & M)[0]      # malformed only
            shared = np.where(A & M)[0]           # both
            union_size = np.sum(A | M)
            jaccard = len(shared) / union_size if union_size > 0 else 0.0

            results.append({
                "lang": lang, "model": model,
                "concept": concept, "layer": lid,
                "n_validity": len(validity),
                "n_error_det": len(error_det),
                "n_shared": len(shared),
                "n_wellformed": int(A.sum()),
                "n_malformed": int(M.sum()),
                "jaccard": jaccard,
                "validity_fraction": len(validity) / int(A.sum()) if A.sum() > 0 else 0.0,
            })

df_results = pd.DataFrame(results)
print(f"Results: {len(df_results)} rows")

if len(df_results) > 0:
    print(f"\nMean Jaccard: {df_results['jaccard'].mean():.3f}")
    print(f"Mean validity fraction: {df_results['validity_fraction'].mean():.3f}")
else:
    print("No results — malformed masks not yet available. Run 1C first.")

In [ ]:
# Cell 4 – Layer profile: validity fraction

if len(df_results) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    for (lang, model), sub in df_results.groupby(["lang", "model"]):
        layer_means = sub.groupby("layer")["validity_fraction"].mean()
        ax.plot(layer_means.index, layer_means.values, label=f"{lang}_{model}", linewidth=1.5)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean validity fraction |V| / |A|")
    ax.set_title("Where is validity encoded?")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, "8_E4_validity_layer_profile.png"), dpi=150)
    plt.show()
else:
    print("No data to plot.")

In [ ]:
# Cell 5 – Save

if len(df_results) > 0:
    out_path = os.path.join(DATA_DIR, "8_E4_wellformedness_results.csv")
    df_results.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

print("\n8 complete.")